# Notebook 04 — The Normal Distribution and the Central Limit Theorem

**Module:** 03 — Statistics and Probability  
**Notebook:** 04 of 18  
**Prerequisites:** NB03  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

The Normal distribution appears everywhere in statistics because of the Central Limit
Theorem (CLT): the sample mean of any distribution converges to Normal as sample size
grows. This is why t-tests, ANOVA, linear regression, and most classical tests work —
they rely on the CLT to justify Normal approximations. Understanding the CLT from
scratch — including what it does and does not say — prevents the most common
statistical errors in biological data analysis.

---
## Step 2 — Intuition

The CLT: roll a die once — uniform distribution. Roll 30 dice and average them — that
average is approximately Normal. Roll 100 dice and average them — even more Normal.
The distribution of the individual measurements doesn't matter; the distribution of
**sample means** is always approximately Normal, for large enough n.

Why? Each die roll is an independent perturbation. Many small independent random
effects, added together, always produce a bell-shaped distribution (by the CLT).
This is why measurement noise in biology is often approximately Normal.

---
## Step 3 — Biological Background

**Where normality holds in biology:**
- Log-transformed expression counts (NB02): approximately normal after log2-CPM
- Gene body GC content: approximately normal across genes
- Many morphological measurements: body weight, height — Normal by CLT (many genes)

**Where normality fails:**
- Raw RNA-seq counts: discrete, overdispersed — use Negative Binomial (DESeq2)
- ChIP-seq peak heights: zero-inflated, heavy-tailed
- Allele frequencies: Beta distribution

**Common distributions in biology:**

| Distribution | Parameters | Biological use |
|-------------|-----------|---------------|
| Normal $N(\mu, \sigma^2)$ | mean, variance | Continuous measurements, log-expression |
| Poisson $\text{Pois}(\lambda)$ | rate λ | Read counts (when mean ≈ variance) |
| Negative Binomial | μ, dispersion | RNA-seq counts (overdispersed) |
| Binomial $\text{Bin}(n, p)$ | n trials, success prob | SNP genotyping, mutation counts |
| Beta $\text{Beta}(\alpha, \beta)$ | shape params | Allele frequencies, methylation β |
| Exponential $\text{Exp}(\lambda)$ | rate | Waiting times, protein half-lives |

---
## Step 4 — Mathematical Explanation

**Normal distribution PDF:**
$$f(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**Standard Normal:** $Z = (X - \mu)/\sigma \sim N(0, 1)$

**68-95-99.7 rule:**
- 68% of data within $\mu \pm \sigma$
- 95% within $\mu \pm 2\sigma$
- 99.7% within $\mu \pm 3\sigma$

**Central Limit Theorem:** if $X_1, X_2, \ldots, X_n$ are i.i.d. with mean $\mu$ and
variance $\sigma^2$, then:
$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{d} N\!\left(\mu, \frac{\sigma^2}{n}\right)$$

The variance of the sample mean is $\sigma^2/n$ — it shrinks as $n$ grows. The
standard error of the mean: $\text{SE} = \sigma/\sqrt{n}$.

---
## Step 5 — Computational Explanation

**`scipy.stats` distribution objects:** each distribution has `.pdf`, `.cdf`, `.ppf`
(percent point function = quantile function, the inverse of CDF), `.rvs` (random
variates), and `.fit` methods.

**Testing normality:**
- Shapiro-Wilk test: `scipy.stats.shapiro(x)` — for n < 5000, the go-to
- Q-Q plot: plot sample quantiles vs. theoretical Normal quantiles — visual

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Normal distribution: PDF, CDF, Z-scores
mu, sigma = 5.0, 1.5
norm = stats.norm(loc=mu, scale=sigma)

x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)

# 68-95-99.7 rule verification
print("68-95-99.7 rule (verification):")
for k, label in [(1, "68.3%"), (2, "95.4%"), (3, "99.7%")]:
    prob = norm.cdf(mu + k*sigma) - norm.cdf(mu - k*sigma)
    print(f"  P(μ ± {k}σ) = {prob*100:.2f}%  (expected {label})")

# Z-score: if GC content ~ N(0.5, 0.08^2), is 0.72 an outlier?
gc_outlier = 0.72
z = (gc_outlier - 0.50) / 0.08
p_tail = 2 * stats.norm.sf(abs(z))  # two-tailed p-value
print(f"\nGC content = {gc_outlier}: Z = {z:.2f}, two-tailed p = {p_tail:.4f}")

In [ ]:
# Cell 6.2 — CLT simulation: sample means converge to Normal
rng = np.random.default_rng(42)

# Parent distribution: Exponential (very non-Normal, right-skewed)
lam = 2.0  # rate; mean = 1/lam, variance = 1/lam^2
parent_mean = 1 / lam
parent_var = 1 / lam**2

n_experiments = 10_000
sample_means = {}

for n in [1, 5, 30, 100]:
    samples = rng.exponential(scale=1/lam, size=(n_experiments, n))
    sample_means[n] = samples.mean(axis=1)
    theoretical_se = np.sqrt(parent_var / n)
    print(f"n={n:3d}: mean of sample means = {sample_means[n].mean():.4f} (true {parent_mean:.4f})"
          f"  SD = {sample_means[n].std():.4f} (expected SE = {theoretical_se:.4f})")

In [ ]:
# Cell 6.3 — Common distributions: Poisson vs. Negative Binomial for counts
mu_count = 50  # mean read count

# Poisson: var = mean
pois_counts = rng.poisson(mu_count, size=5000)

# Negative Binomial: var = mean + mean^2 / dispersion
# scipy parametrisation: n=dispersion, p=n/(n+mu)
dispersion = 3.0
nb_counts = rng.negative_binomial(n=dispersion, p=dispersion/(dispersion+mu_count), size=5000)

print(f"Poisson (μ={mu_count}): mean={pois_counts.mean():.1f}, var={pois_counts.var():.1f}")
print(f"Neg.Bin. (μ={mu_count}, disp={dispersion}): mean={nb_counts.mean():.1f}, var={nb_counts.var():.1f}")
print(f"\nRNA-seq reality: variance >> mean → Negative Binomial is correct model")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — CLT in action + distribution gallery
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Panel 1: CLT convergence
ax = axes[0, 0]
colors = ["gray", "orange", "steelblue", "tomato"]
x_plot = np.linspace(-1, 3, 400)
for (n, means), color in zip(sample_means.items(), colors):
    ax.hist(means, bins=60, density=True, alpha=0.5, color=color, label=f"n={n}")
    # Theoretical Normal
    se = np.sqrt(parent_var / n)
    ax.plot(x_plot, stats.norm.pdf(x_plot, parent_mean, se), color=color, lw=1.5)
ax.set_xlim(-0.5, 2.5)
ax.set_xlabel("Sample mean"); ax.set_ylabel("Density")
ax.set_title("CLT: Exponential parent → Normal sample means")
ax.legend(fontsize=8)

# Panel 2: Normal distribution with 68-95-99.7 bands
ax = axes[0, 1]
x_n = np.linspace(-4, 4, 500)
ax.plot(x_n, stats.norm.pdf(x_n), color="black", lw=2)
for k, color in [(1, "#d4e6f1"), (2, "#85c1e9"), (3, "#2e86c1")]:
    ax.fill_between(x_n, stats.norm.pdf(x_n),
                    where=np.abs(x_n) <= k, color=color, alpha=0.7)
    prob = 2 * stats.norm.cdf(k) - 1
    ax.text(k*0.7, stats.norm.pdf(k*0.7)/2, f"{prob*100:.1f}%", ha="center", fontsize=8)
ax.set_xlabel("Z-score"); ax.set_ylabel("Density")
ax.set_title("Normal: 68-95-99.7 rule")

# Panel 3: Poisson vs. Negative Binomial
ax = axes[1, 0]
bins = np.arange(0, 200, 4)
ax.hist(pois_counts, bins=bins, density=True, alpha=0.6, color="steelblue", label=f"Poisson(μ={mu_count})")
ax.hist(nb_counts, bins=bins, density=True, alpha=0.6, color="tomato", label=f"NegBin(μ={mu_count}, d={dispersion})")
ax.set_xlabel("Count"); ax.set_ylabel("Density")
ax.set_title("Count distributions (RNA-seq)")
ax.legend(fontsize=8)

# Panel 4: QQ-plot for normality check
ax = axes[1, 1]
log_data = rng.lognormal(2, 0.5, 200)
log_log_data = np.log(log_data)
stats.probplot(log_log_data, dist="norm", plot=ax)
ax.set_title("QQ-plot: log-transformed lognormal is Normal")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Simulate the CLT using a **Uniform(0,1)** parent distribution: draw 10,000 samples
   of size n=1, 5, 30, 100. Plot the sample mean distributions. At what n does the
   distribution look approximately Normal by eye?
2. Gene expression for a gene follows approximately $N(\mu=8, \sigma=1.2)$ on the log2
   scale. What fraction of samples would you expect to have expression > 10?
   Compute using `scipy.stats.norm.sf` and verify by simulation.
3. Why is the Negative Binomial better than the Poisson for RNA-seq count data?
   Write two sentences, then compute the variance of Poisson(50) vs. NegBin(50, disp=3)
   analytically.
4. Implement a Q-Q plot function from scratch (no `scipy.stats.probplot`):
   sort your data, compute the theoretical quantiles of $N(0,1)$ at evenly-spaced
   probability points, and scatter-plot them. Test on normal data (should be straight
   line) and on exponential data (should curve).

---
## Quiz — Active Recall

1. State the Central Limit Theorem in plain English, no equations.
2. Write the Normal PDF. Define every symbol.
3. What is the standard error of the mean? How does it change with sample size?
4. Name four distributions used in biology. Match each to a use case.
5. Why does RNA-seq data require Negative Binomial, not Poisson?

---
## Reflection

**Date completed:** ____________________

> *[Can you state the CLT and explain why it justifies using t-tests on log-expression data?]*

---
*Next: `05_hypothesis_testing_fundamentals.ipynb`*